In [2]:
# --- 1. Imports and Setup ---
import os
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil
from sklearn.metrics import r2_score as sk_r2_score
from google.colab import drive
import kagglehub
import warnings
import time
import tensorflow.keras.backend as K # Import Keras backend

# Suppress TensorFlow/Keras warnings
warnings.filterwarnings("ignore", category=UserWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

# --- 2. Mount Google Drive ---
try:
    drive.mount('/content/drive', force_remount=True)
    print("--- Drive Mount Successful ---")
except Exception as e:
    print(f"--- ERROR Mounting Drive: {e} ---")
    raise SystemExit("Drive mounting failed. Cannot proceed.")

# --- 3. Connect to Kaggle Datasets ---
print("Connecting to Kaggle datasets...")
try:
    syxlicheng_heartdatabase_path = kagglehub.dataset_download('syxlicheng/heartdatabase')
    shravansaranyan_lvef_two_stream_tfrecord_files_path = kagglehub.dataset_download('shravansaranyan/lvef-two-stream-tfrecord-files')
    shravansaranyan_lvef_i3d_regression_tfrecord_files_path = kagglehub.dataset_download('shravansaranyan/lvef-i3d-regression-tfrecord-files')
    shravansaranyan_lvef_i3d_regression_tfrecord_v_s_files_path = kagglehub.dataset_download('shravansaranyan/lvef-i3d-regression-tfrecord-v-s-files')
    print("Dataset connection complete.")
except Exception as e:
    print(f"--- ERROR Connecting to Kaggle Datasets: {e} ---")
    raise SystemExit("Kaggle connection failed. Cannot proceed.")

# --- 4. Define Paths and Constants ---
MODELS_BASE_DIR = "/content/drive/My Drive/LVEF Prediction Models"
CSV_PATH = os.path.join(syxlicheng_heartdatabase_path, 'EchoNet-Dynamic', 'FileList.csv')
BATCH_SIZE_I3D = 2
BATCH_SIZE_TWO_STREAM = 16
BATCH_SIZE_DI_FUSION = 4
BATCH_SIZE_SEGMENTATION = 2
IMG_HEIGHT, IMG_WIDTH, NUM_FRAMES = 112, 112, 28
SIZE = 112 # Explicitly define for segmentation loader
STATE_FILE_PATH = "/content/drive/My Drive/lvef_data_export_progress.txt"

# --- 5. Reusable Helper Functions ---

# --- State Management ---
def load_completed_folders(state_file):
    """Reads the state file and returns a set of completed folder paths."""
    completed = set()
    if os.path.exists(state_file):
        try:
            with open(state_file, 'r') as f:
                for line in f:
                    completed.add(line.strip())
            print(f"Loaded {len(completed)} completed folders from state file.")
        except Exception as e:
            print(f"WARNING: Could not read state file {state_file}. Error: {e}")
    else:
        print("No state file found. Starting fresh.")
    return completed

def save_completed_folder(state_file, folder_path):
    """Appends a successfully completed folder path to the state file."""
    try:
        with open(state_file, 'a') as f:
            f.write(folder_path + '\n')
    except Exception as e:
        print(f"WARNING: Could not write to state file {state_file}. Error: {e}")

# --- Custom Metrics/Losses (Defined Globally and Registered) ---
@tf.keras.utils.register_keras_serializable()
def r2_score(y_true, y_pred):
    """Custom Keras R^2 metric."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    ss_res = K.sum(K.square(y_true - y_pred))
    ss_tot = K.sum(K.square(y_true - K.mean(y_true)))
    return K.switch(K.equal(ss_tot, 0), K.zeros_like(ss_tot), 1 - ss_res / (ss_tot + K.epsilon()))

@tf.keras.utils.register_keras_serializable()
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """Dice coefficient metric for segmentation."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

@tf.keras.utils.register_keras_serializable()
def dice_loss(y_true, y_pred):
    """Dice loss function."""
    return 1.0 - dice_coefficient(y_true, y_pred)

@tf.keras.utils.register_keras_serializable()
def dice_bce_loss(y_true, y_pred):
    """Combined BCE and Dice loss."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred, from_logits=False)
    d_loss = dice_loss(y_true, y_pred)
    return bce + d_loss
# --- End Custom Metrics/Losses ---

# --- Dataset Creation Functions ---
def create_two_stream_dataset(tfrecord_path, batch_size):
    """Loads and parses the standard Two-Stream TFRecord dataset."""
    ds = tf.data.TFRecordDataset(tfrecord_path, num_parallel_reads=tf.data.AUTOTUNE)
    _feature_desc = {
        'clip_raw': tf.io.FixedLenFeature([], tf.string),
        'label_ef': tf.io.FixedLenFeature([], tf.float32, default_value=-1.0),
        'ef_label': tf.io.FixedLenFeature([], tf.float32, default_value=-1.0),
    }
    def _parse_and_split(proto):
        example = tf.io.parse_single_example(proto, _feature_desc)
        clip = tf.io.parse_tensor(example['clip_raw'], out_type=tf.float32)
        clip = tf.ensure_shape(clip, (NUM_FRAMES, IMG_HEIGHT, IMG_WIDTH))
        clip = tf.expand_dims(clip, axis=-1)
        label = tf.cond(tf.not_equal(example['ef_label'], -1.0), lambda: example['ef_label'], lambda: example['label_ef'])
        spatial_stream = clip[:, :, :IMG_WIDTH // 2, :]
        temporal_half = clip[:, :, IMG_WIDTH // 2:, :]
        temporal_stream = temporal_half[1:, :, :, :] - temporal_half[:-1, :, :, :]
        temporal_mean = tf.reduce_mean(temporal_stream, axis=[1, 2, 3], keepdims=True)
        temporal_std = tf.math.reduce_std(temporal_stream, axis=[1, 2, 3], keepdims=True)
        temporal_stream = (temporal_stream - temporal_mean) / tf.maximum(temporal_std, 1e-6)
        return (spatial_stream, temporal_stream), label
    ds = ds.map(_parse_and_split, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def create_i3d_dataset(tfrecord_path, batch_size):
    """Loads and parses the 1-channel I3D TFRecord dataset."""
    ds = tf.data.TFRecordDataset(tfrecord_path, num_parallel_reads=tf.data.AUTOTUNE)
    _feature_desc = {
        'clip_raw': tf.io.FixedLenFeature([], tf.string),
        'label_ef': tf.io.FixedLenFeature([], tf.float32, default_value=-1.0),
        'ef_label': tf.io.FixedLenFeature([], tf.float32, default_value=-1.0),
    }
    def _parse_full_clip(proto):
        example = tf.io.parse_single_example(proto, _feature_desc)
        clip = tf.io.parse_tensor(example['clip_raw'], out_type=tf.float32)
        clip = tf.ensure_shape(clip, (NUM_FRAMES, IMG_HEIGHT, IMG_WIDTH))
        clip = tf.expand_dims(clip, axis=-1)
        label = tf.cond(tf.not_equal(example['ef_label'], -1.0), lambda: example['ef_label'], lambda: example['label_ef'])
        return clip, label
    ds = ds.map(_parse_full_clip, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def create_i3d_dataset_3_channel(tfrecord_path, batch_size):
    """Loads and parses the 3-channel I3D TFRecord dataset."""
    ds = tf.data.TFRecordDataset(tfrecord_path, num_parallel_reads=tf.data.AUTOTUNE)
    _feature_desc = {
        'clip_raw': tf.io.FixedLenFeature([], tf.string),
        'label_ef': tf.io.FixedLenFeature([], tf.float32, default_value=-1.0),
        'ef_label': tf.io.FixedLenFeature([], tf.float32, default_value=-1.0),
    }
    def _parse_full_clip(proto):
        example = tf.io.parse_single_example(proto, _feature_desc)
        clip = tf.io.parse_tensor(example['clip_raw'], out_type=tf.float32)
        clip = tf.ensure_shape(clip, (NUM_FRAMES, IMG_HEIGHT, IMG_WIDTH))
        clip = tf.expand_dims(clip, axis=-1)
        clip = tf.repeat(clip, 3, axis=-1)
        label = tf.cond(tf.not_equal(example['label_ef'], -1.0), lambda: example['label_ef'], lambda: example['ef_label'])
        return clip, label
    ds = ds.map(_parse_full_clip, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def create_di_fusion_dataset(tfrecord_path, batch_size):
    """Loads and prepares data specifically for the DI/DIT Fusion models."""
    ds = tf.data.TFRecordDataset(tfrecord_path, num_parallel_reads=tf.data.AUTOTUNE)
    di_feature_description = { # DI uses ef_label
        'clip_raw': tf.io.FixedLenFeature([], tf.string),
        'ef_label': tf.io.FixedLenFeature([], tf.float32),
    }
    def _parse_full_clip(proto):
        example = tf.io.parse_single_example(proto, di_feature_description)
        clip = tf.io.parse_tensor(example['clip_raw'], out_type=tf.float32)
        clip = tf.ensure_shape(clip, (NUM_FRAMES, IMG_HEIGHT, IMG_WIDTH))
        clip = tf.expand_dims(clip, axis=-1)
        label = example['ef_label']
        return clip, label
    ds = ds.map(_parse_full_clip, num_parallel_calls=tf.data.AUTOTUNE)

    def _generate_streams_dict(clip, label):
        spatial_stream = clip
        temporal_stream = clip[1:, :, :, :] - clip[:-1, :, :, :]
        mean = tf.reduce_mean(temporal_stream)
        std = tf.math.reduce_std(temporal_stream)
        temporal_stream_normalized = (temporal_stream - mean) / tf.maximum(std, 1e-6)
        temporal_stream_final = tf.ensure_shape(temporal_stream_normalized, (NUM_FRAMES-1, IMG_HEIGHT, IMG_WIDTH, 1))
        return {'spatial_input': spatial_stream, 'temporal_input': temporal_stream_final}, label

    ds = ds.map(_generate_streams_dict, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

def create_segmentation_dataset(tfrecord_path, batch_size):
    """Loads and parses data from the multi-output TFRecords for segmentation models."""
    ds = tf.data.TFRecordDataset(tfrecord_path, num_parallel_reads=tf.data.AUTOTUNE)
    seg_feature_description = {
        'clip_raw': tf.io.FixedLenFeature([], tf.string),
        'label_ef': tf.io.FixedLenFeature([], tf.float32),
        'label_mask': tf.io.FixedLenFeature([], tf.string),
    }

    # Capture global constants
    local_frames = NUM_FRAMES
    local_size_h = IMG_HEIGHT
    local_size_w = IMG_WIDTH

    def _parse_multi_output(proto):
        parsed_example = tf.io.parse_single_example(proto, seg_feature_description)
        clip = tf.io.parse_tensor(parsed_example['clip_raw'], out_type=tf.float32)
        clip = tf.reshape(clip, [local_frames, local_size_h, local_size_w, 1])
        label_ef = parsed_example['label_ef']
        single_mask = tf.io.parse_tensor(parsed_example['label_mask'], out_type=tf.float32)
        expanded_mask = tf.expand_dims(single_mask, axis=0)
        label_mask = tf.tile(expanded_mask, [local_frames, 1, 1, 1])
        label_mask = tf.ensure_shape(label_mask, (local_frames, local_size_h, local_size_w, 1))
        labels = {"ef_output": label_ef, "seg_output": label_mask}
        return clip, labels

    ds = ds.map(_parse_multi_output, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

def generate_history_plot(experiment_dir, run_name):
    """Generates the training history plot with the correct title, if it doesn't exist."""
    print(f"INFO: Checking for Training History Plot for [{run_name}]...")
    history_plot_path = os.path.join(experiment_dir, "training_history_plot.png")

    if os.path.exists(history_plot_path):
        print(f"INFO: 'training_history_plot.png' already exists. Skipping generation.")
        return True

    print(f"INFO: Plot not found. Generating...")

    # --- FIX: Corrected run_name check based on last log ---
    if run_name == "CNN_RNN T(O)-a":
        primary_log = "rnn_training_log.csv"
    else:
        primary_log = "training_log.csv"
    # --- END FIX ---

    fallback_log = "final_history.csv"
    log_files_to_try = [primary_log, fallback_log]

    history_df = None
    log_file_used = ""
    for log_filename in log_files_to_try:
        log_file_path = os.path.join(experiment_dir, log_filename)
        if not os.path.exists(log_file_path):
            continue
        try:
            df = pd.read_csv(log_file_path)
            loss_col, val_loss_col = None, None
            if 'loss' in df.columns and 'val_loss' in df.columns:
                loss_col, val_loss_col = 'loss', 'val_loss'
            elif 'ef_output_loss' in df.columns and 'val_ef_output_loss' in df.columns:
                 loss_col, val_loss_col = 'ef_output_loss', 'val_ef_output_loss'
                 print(f"INFO: Using '{loss_col}'/'{val_loss_col}' columns from {log_filename}")

            if loss_col and val_loss_col:
                history_df = df.rename(columns={loss_col: 'loss', val_loss_col: 'val_loss'})
                log_file_used = log_filename
                print(f"INFO: Found valid log file: '{log_file_used}'")
                break
        except Exception as e:
             print(f"INFO: Could not parse log file '{log_filename}'. Reason: {e}. Trying next file.")
             continue
    if history_df is None:
        print(f"WARNING: No valid log file found. Cannot generate plot.")
        return False

    try:
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        x_axis = history_df.index
        ax.plot(x_axis, history_df['loss'], label='Training Loss (EF or Total)')
        ax.plot(x_axis, history_df['val_loss'], label='Validation Loss (EF or Total)')
        ax.set_title(f'Training History for {run_name}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss (MSE or Combined)')
        ax.legend()
        ax.grid(True)
        fig.savefig(history_plot_path)
        plt.close(fig)
        print(f"SUCCESS: Generated 'training_history_plot.png' (from {log_file_used}).")
        return True
    except Exception as e:
        print(f"ERROR: Could not generate history plot for '{run_name}'. Reason: {e}")
        return False

# --- 6. Main Resumable Data Export Loop ---
print("\n" + "="*50)
print("STARTING NON-DESTRUCTIVE RESUMABLE DATA EXPORT")
print("This will ONLY generate files that are missing.")
print(f"Progress tracked in: {STATE_FILE_PATH}")
print("="*50 + "\n")

completed_folders = load_completed_folders(STATE_FILE_PATH)
model_folders_to_process = []
print("Searching for model folders containing 'best_model.keras'...")
for root, dirs, files in os.walk(MODELS_BASE_DIR):
    # Check for both possible names
    best_model_file = None
    if "best_model.keras" in files:
        best_model_file = "best_model.keras"
    elif "best_i3d_model.keras" in files:
        best_model_file = "best_i3d_model.keras"

    if best_model_file:
        best_model_path = os.path.join(root, "best_model.keras")
        # If the standard name doesn't exist, create it by copying
        if not os.path.exists(best_model_path) and best_model_file == "best_i3d_model.keras":
            try:
                alt_model_path = os.path.join(root, "best_i3d_model.keras")
                print(f"INFO: Found 'best_i3d_model.keras' in {root}. Copying to 'best_model.keras'.")
                shutil.copy(alt_model_path, best_model_path)
            except Exception as e:
                print(f"WARNING: Could not copy 'best_i3d_model.keras' in {root}. Error: {e}")

        # Only add if 'best_model.keras' now exists
        if os.path.exists(best_model_path):
             model_folders_to_process.append(root)

        dirs[:] = [] # Prevent descending further
model_folders_to_process.sort()

if not model_folders_to_process: print("WARNING: No models found."); raise SystemExit("No models found.")
else: print(f"Found {len(model_folders_to_process)} total model folder(s).")

# --- FIX: Corrected problematic model name sets based on logs ---
problematic_fusion_names = {
    "Fusion DI-B1", "Fusion DI-B2", "Fusion DI-L1","Fusion DI-L2",
    "Fusion DIT-1", "Fusion DIT-2", "Fusion DI-2"
}
segmentation_model_names = {"I3D Segmentation M-1", "I3D Segmentation M-2", "I3D Segmentation M-3"}
# --- END FIX ---

# --- FIX: CORRECTED COUNTER INITIALIZATION ---
total_folders = len(model_folders_to_process)
processed_in_this_run = 0
newly_completed = 0
skipped_already_done = 0
error_count_this_run = 0
# --- END FIX ---

for i, experiment_dir in enumerate(model_folders_to_process):
    run_name = os.path.basename(experiment_dir)
    print(f"\n--- Checking Folder {i+1}/{total_folders}: [{run_name}] ---")

    if experiment_dir in completed_folders: print("INFO: Skipping folder - already marked as completed."); skipped_already_done += 1; continue

    processed_in_this_run += 1
    print(f"Attempting processing for [{run_name}]...")

    best_model_path = os.path.join(experiment_dir, "best_model.keras"); metrics_path = os.path.join(experiment_dir, "evaluation_metrics.txt")
    plot_path = os.path.join(experiment_dir, "performance_analysis_plot.png"); history_plot_path = os.path.join(experiment_dir, "training_history_plot.png")
    y_true_path = os.path.join(experiment_dir, 'y_test_true.npy'); y_pred_path = os.path.join(experiment_dir, 'y_test_pred.npy')
    y_true_val_path = os.path.join(experiment_dir, 'y_val_true.npy'); y_pred_val_path = os.path.join(experiment_dir, 'y_val_pred.npy')
    t_o_a_test_features_path = os.path.join(experiment_dir, 'test_features.npy'); t_o_a_test_labels_path = os.path.join(experiment_dir, 'test_labels.npy')
    t_o_a_val_features_path = os.path.join(experiment_dir, 'val_features.npy'); t_o_a_val_labels_path = os.path.join(experiment_dir, 'val_labels.npy')

    folder_successfully_processed = False
    val_metrics = {'rmse': np.nan, 'mae': np.nan}; test_metrics = {'rmse': np.nan, 'mae': np.nan}
    val_r2_sklearn = np.nan; test_r2_sklearn = np.nan
    y_test_true = None; y_test_pred = None; y_val_true = None; y_val_pred = None

    try:
        generate_history_plot(experiment_dir, run_name) # Non-destructive
        if not os.path.exists(best_model_path): raise FileNotFoundError(f"best_model.keras missing in {run_name}")

        evaluation_files_missing = (
            not os.path.exists(metrics_path) or
            not os.path.exists(plot_path) or
            not os.path.exists(y_true_path) or
            not os.path.exists(y_pred_path) or
            not os.path.exists(y_true_val_path) or
            not os.path.exists(y_pred_val_path)
        )

        if not evaluation_files_missing:
            print("INFO: All evaluation files (.npy, .txt, .png) already exist. Skipping evaluation.")
            folder_successfully_processed = True
            continue

        print("INFO: One or more evaluation files missing. Running evaluation...")

        # --- FIX: Corrected name for T(O)-a ---
        if run_name == "CNN_RNN T(O)-a":
            print("\nINFO: Handling 'CNN_RNN T(O)-a' using saved .npy features.")
            required_npy = [t_o_a_test_features_path, t_o_a_test_labels_path, t_o_a_val_features_path, t_o_a_val_labels_path]
            if not all(os.path.exists(p) for p in required_npy): raise FileNotFoundError(f"Missing feature .npy files in {run_name}")
            try: model = tf.keras.models.load_model(best_model_path, custom_objects={'r2_score': r2_score}, safe_mode=False)
            except Exception as e: print(f"ERROR loading RNN model {run_name}: {e}"); raise
            try:
                print("INFO: Loading .npy features/labels..."); test_features = np.load(t_o_a_test_features_path); test_labels = np.load(t_o_a_test_labels_path)
                val_features = np.load(t_o_a_val_features_path); val_labels = np.load(t_o_a_val_labels_path)
                test_labels_flat = test_labels.flatten(); val_labels_flat = val_labels.flatten()
                y_val_true = val_labels_flat; y_test_true = test_labels_flat
            except Exception as e: print(f"ERROR loading feature .npy files: {e}"); raise
            try:
                print("INFO: Evaluating validation features..."); val_metrics_raw = model.evaluate(val_features, val_labels_flat, batch_size=BATCH_SIZE_I3D, return_dict=True, verbose=0); val_metrics = {'rmse': val_metrics_raw.get('rmse', np.nan), 'mae': val_metrics_raw.get('mae', np.nan)}
                print("INFO: Evaluating test features..."); test_metrics_raw = model.evaluate(test_features, test_labels_flat, batch_size=BATCH_SIZE_I3D, return_dict=True, verbose=0); test_metrics = {'rmse': test_metrics_raw.get('rmse', np.nan), 'mae': test_metrics_raw.get('mae', np.nan)}
                print("INFO: Predicting validation features..."); y_val_pred = model.predict(val_features, batch_size=BATCH_SIZE_I3D, verbose=0); val_r2_sklearn = sk_r2_score(val_labels_flat, y_val_pred)
                print("INFO: Predicting test features..."); y_test_pred = model.predict(test_features, batch_size=BATCH_SIZE_I3D, verbose=0); test_r2_sklearn = sk_r2_score(test_labels_flat, y_test_pred)
                if not os.path.exists(y_true_val_path): np.save(y_true_val_path, val_labels_flat)
                if not os.path.exists(y_pred_val_path): np.save(y_pred_val_path, y_val_pred)
                if not os.path.exists(y_true_path): np.save(y_true_path, test_labels_flat)
                if not os.path.exists(y_pred_path): np.save(y_pred_path, y_test_pred)
                print("SUCCESS: Eval complete, results saved.")
            except Exception as e: print(f"ERROR during feature evaluation/prediction: {e}"); raise
            folder_successfully_processed = True

        # --- Handling for Segmentation Models ---
        elif run_name in segmentation_model_names:
            print(f"\nINFO: Handling Segmentation model '{run_name}'.")
            is_m3 = (run_name == "I3D Segmentation M-3")
            compile_model = not is_m3
            try:
                print(f"Loading model with compile={compile_model}...")
                custom_objects_seg = {'r2_score': r2_score,'dice_coefficient': dice_coefficient,'dice_loss': dice_loss,'dice_bce_loss': dice_bce_loss}
                model = tf.keras.models.load_model(best_model_path, custom_objects=custom_objects_seg, compile=compile_model, safe_mode=False)
            except Exception as e: print(f"ERROR loading model {run_name}: {e}"); raise

            print("Using Segmentation multi-output loader."); batch_size = BATCH_SIZE_SEGMENTATION
            tf_path_base = shravansaranyan_lvef_i3d_regression_tfrecord_v_s_files_path
            try: val_ds = create_segmentation_dataset(os.path.join(tf_path_base, 'val_multi_output.tfrecord'), batch_size); test_ds = create_segmentation_dataset(os.path.join(tf_path_base, 'test_multi_output.tfrecord'), batch_size)
            except Exception as e: print(f"ERROR during segmentation dataset creation: {e}"); raise
            if val_ds is None or test_ds is None: raise ValueError("Segmentation dataset loading failed")

            try: df = pd.read_csv(CSV_PATH); val_steps = -(-len(df[df["Split"] == "VAL"]) // batch_size); test_steps = -(-len(df[df["Split"] == "TEST"]) // batch_size)
            except Exception as e: print(f"ERROR reading CSV or calculating steps: {e}"); raise

            try: # Skip Evaluate, only Predict
                print("INFO: Skipping model.evaluate. Predicting and calculating metrics manually.")
                print(f"INFO: Predicting validation set...")
                y_val_true_list = [y['ef_output'] for x, y in val_ds.take(val_steps)]; y_val_pred_raw = model.predict(val_ds.take(val_steps), verbose=0)
                y_val_true = np.concatenate(y_val_true_list)
                if isinstance(y_val_pred_raw, dict) and 'ef_output' in y_val_pred_raw: y_val_pred = y_val_pred_raw['ef_output']
                else: raise ValueError(f"Could not find 'ef_output' in val predictions for {run_name}")
                if isinstance(y_val_pred, list): y_val_pred = np.concatenate(y_val_pred)
                y_val_true = y_val_true.flatten(); y_val_pred = y_val_pred.flatten()
                if len(y_val_true) != len(y_val_pred): print(f"WARN Val shapes: {y_val_true.shape} vs {y_val_pred.shape}"); y_val_pred = y_val_pred[:len(y_val_true)] if len(y_val_pred) > len(y_val_true) else np.resize(y_val_pred, y_val_true.shape)
                val_r2_sklearn = sk_r2_score(y_val_true, y_val_pred)
                val_metrics['rmse'] = np.sqrt(np.mean((y_val_true - y_val_pred)**2))
                val_metrics['mae'] = np.mean(np.abs(y_val_true - y_val_pred))

                print(f"INFO: Predicting test set...")
                y_test_true_list = [y['ef_output'] for x, y in test_ds.take(test_steps)]; y_test_pred_raw = model.predict(test_ds.take(test_steps), verbose=0)
                y_test_true = np.concatenate(y_test_true_list)
                if isinstance(y_test_pred_raw, dict) and 'ef_output' in y_test_pred_raw: y_test_pred = y_test_pred_raw['ef_output']
                else: raise ValueError(f"Could not find 'ef_output' in test predictions for {run_name}")
                if isinstance(y_test_pred, list): y_test_pred = np.concatenate(y_test_pred)
                y_test_true = y_test_true.flatten(); y_test_pred = y_test_pred.flatten()
                if len(y_test_true) != len(y_test_pred): print(f"WARN Test shapes: {y_test_true.shape} vs {y_test_pred.shape}"); y_test_pred = y_test_pred[:len(y_test_true)] if len(y_test_pred) > len(y_test_true) else np.resize(y_test_pred, y_test_true.shape)
                test_r2_sklearn = sk_r2_score(y_test_true, y_test_pred)
                test_metrics['rmse'] = np.sqrt(np.mean((y_test_true - y_test_pred)**2))
                test_metrics['mae'] = np.mean(np.abs(y_test_true - y_test_pred))

                if not os.path.exists(y_true_val_path): np.save(y_true_val_path, y_val_true)
                if not os.path.exists(y_pred_val_path): np.save(y_pred_val_path, y_val_pred)
                if not os.path.exists(y_true_path): np.save(y_true_path, y_test_true)
                if not os.path.exists(y_pred_path): np.save(y_pred_path, y_test_pred)
                print("SUCCESS: Prediction complete and EF results saved.")
            except Exception as e: print(f"ERROR during segmentation model prediction/extraction: {e}"); raise
            folder_successfully_processed = True

        # --- REGULAR EVALUATION FOR ALL OTHER MODELS ---
        else:
            print("INFO: Proceeding with standard evaluation...")
            run_name_lower = run_name.lower(); val_ds, test_ds = None, None; batch_size = 0
            try: # Corrected Dataset Loading
                if run_name in problematic_fusion_names:
                    print(f"INFO: Detected problematic Fusion model [{run_name}]. Using CORRECT DI Fusion loader.")
                    batch_size = BATCH_SIZE_DI_FUSION; tf_path_base = shravansaranyan_lvef_two_stream_tfrecord_files_path
                    val_ds = create_di_fusion_dataset(os.path.join(tf_path_base, 'val.tfrecord'), batch_size)
                    test_ds = create_di_fusion_dataset(os.path.join(tf_path_base, 'test.tfrecord'), batch_size)
                elif ('fusion' in run_name_lower and ' si' not in run_name_lower) or 'two-stream' in run_name_lower:
                    print(f"INFO: Detected standard Two-Stream/Fusion model.")
                    batch_size = BATCH_SIZE_TWO_STREAM; tf_path_base = shravansaranyan_lvef_two_stream_tfrecord_files_path
                    val_ds = create_two_stream_dataset(os.path.join(tf_path_base, 'val.tfrecord'), batch_size)
                    test_ds = create_two_stream_dataset(os.path.join(tf_path_base, 'test.tfrecord'), batch_size)
                else: # I3D, CNN_RNN (non-T(O)-a), SI Fusion
                    print(f"INFO: Detected standard Single-Stream model type.")
                    batch_size = BATCH_SIZE_I3D; tf_path_base = shravansaranyan_lvef_i3d_regression_tfrecord_files_path
                    if 'cnn_rnn' in run_name_lower and 'scratch' in run_name_lower: print("INFO: Using 1-channel I3D loader ('scratch')."); val_ds = create_i3d_dataset(os.path.join(tf_path_base, 'val.tfrecord'), batch_size); test_ds = create_i3d_dataset(os.path.join(tf_path_base, 'test.tfrecord'), batch_size)
                    elif 'cnn_rnn' in run_name_lower: print("INFO: Using 3-channel I3D loader ('cnn_rnn' pretrained)."); val_ds = create_i3d_dataset_3_channel(os.path.join(tf_path_base, 'val.tfrecord'), batch_size); test_ds = create_i3d_dataset_3_channel(os.path.join(tf_path_base, 'test.tfrecord'), batch_size)
                    else: print("INFO: Using 1-channel I3D loader (standard/SI)."); val_ds = create_i3d_dataset(os.path.join(tf_path_base, 'val.tfrecord'), batch_size); test_ds = create_i3d_dataset(os.path.join(tf_path_base, 'test.tfrecord'), batch_size)
            except Exception as e: print(f"ERROR during dataset creation: {e}"); raise
            if val_ds is None or test_ds is None or batch_size == 0: raise ValueError("Dataset loading failed")

            try: model = tf.keras.models.load_model(best_model_path, custom_objects={'r2_score': r2_score}, safe_mode=False)
            except Exception as e: print(f"ERROR loading model: {e}"); raise
            try: df = pd.read_csv(CSV_PATH); val_steps = -(-len(df[df["Split"] == "VAL"]) // batch_size); test_steps = -(-len(df[df["Split"] == "TEST"]) // batch_size)
            except Exception as e: print(f"ERROR reading CSV or calculating steps: {e}"); raise

            try:
                print(f"INFO: Evaluating validation set..."); val_metrics = model.evaluate(val_ds, steps=val_steps, return_dict=True, verbose=0)
                print(f"INFO: Evaluating test set..."); test_metrics = model.evaluate(test_ds, steps=test_steps, return_dict=True, verbose=0)
                print(f"INFO: Predicting validation set...")
                if isinstance(val_ds.element_spec[0], dict): y_val_true_list = [y for x, y in val_ds.take(val_steps)]; y_val_pred_list = model.predict(val_ds.take(val_steps), verbose=0)
                else: y_val_true_list = [y for x, y in val_ds.take(val_steps)]; y_val_pred_list = model.predict(val_ds.take(val_steps), verbose=0)
                y_val_true = np.concatenate(y_val_true_list); y_val_pred = np.concatenate(y_val_pred_list)
                if len(y_val_true) != len(y_val_pred): print(f"WARN Val shapes: {y_val_true.shape} vs {y_val_pred.shape}"); y_val_pred = y_val_pred[:len(y_val_true)] if len(y_val_pred) > len(y_val_true) else np.resize(y_val_pred, y_val_true.shape)
                val_r2_sklearn = sk_r2_score(y_val_true, y_val_pred)

                print(f"INFO: Predicting test set...")
                if isinstance(test_ds.element_spec[0], dict): y_test_true_list = [y for x, y in test_ds.take(test_steps)]; y_test_pred_list = model.predict(test_ds.take(test_steps), verbose=0)
                else: y_test_true_list = [y for x, y in test_ds.take(test_steps)]; y_test_pred_list = model.predict(test_ds.take(test_steps), verbose=0)
                y_test_true = np.concatenate(y_test_true_list); y_test_pred = np.concatenate(y_test_pred_list)
                if len(y_test_true) != len(y_test_pred): print(f"WARN Test shapes: {y_test_true.shape} vs {y_test_pred.shape}"); y_test_pred = y_test_pred[:len(y_test_true)] if len(y_test_pred) > len(y_test_true) else np.resize(y_test_pred, y_test_true.shape)
                test_r2_sklearn = sk_r2_score(y_test_true, y_test_pred)
            except Exception as e: print(f"ERROR during evaluation or prediction: {e}"); raise

            try: # Non-destructive .npy saving
                if not os.path.exists(y_true_val_path): np.save(y_true_val_path, y_val_true)
                if not os.path.exists(y_pred_val_path): np.save(y_pred_val_path, y_val_pred)
                if not os.path.exists(y_true_path): np.save(y_true_path, y_test_true)
                if not os.path.exists(y_pred_path): np.save(y_pred_path, y_test_pred)
                print("SUCCESS: Model evaluation complete and prediction data saved.")
            except Exception as e: print(f"ERROR saving .npy files: {e}"); raise

            folder_successfully_processed = True

        # --- Common block for ALL models ---
        if not os.path.exists(plot_path): # Only generate if missing
            print(f"INFO: 'performance_analysis_plot.png' missing. Generating...")
            try:
                # --- FIX: Corrected name for T(O)-a ---
                if run_name == "CNN_RNN T(O)-a" or run_name in segmentation_model_names: y_test_true_plot, y_test_pred_plot = y_test_true, y_test_pred
                else: y_test_true_plot, y_test_pred_plot = y_test_true, y_test_pred

                if y_test_true_plot is None or y_test_pred_plot is None or len(y_test_true_plot) == 0: print("WARNING: No valid prediction data. Skipping performance plot.")
                else:
                    fig, axes = plt.subplots(1, 2, figsize=(16, 6)); y_true_flat=y_test_true_plot.flatten(); y_pred_flat=y_test_pred_plot.flatten()
                    if len(y_true_flat) > 0 and len(y_pred_flat) > 0:
                        axes[0].scatter(y_true_flat, y_pred_flat, alpha=0.5); min_val=min(np.min(y_true_flat), np.min(y_pred_flat)); max_val=max(np.max(y_true_flat), np.max(y_pred_flat))
                        axes[0].plot([min_val, max_val], [min_val, max_val], '--', lw=2, color='red'); axes[0].set_xlabel("Ground Truth EF (%)"); axes[0].set_ylabel("Predicted EF (%)"); axes[0].set_title("Prediction vs. Ground Truth"); axes[0].grid(True); axes[0].set_aspect('equal', adjustable='box')
                        mean_ef=np.mean([y_true_flat, y_pred_flat], axis=0); diff_ef=y_true_flat - y_pred_flat; md=np.mean(diff_ef); sd=np.std(diff_ef)
                        axes[1].scatter(mean_ef, diff_ef, alpha=0.5); axes[1].axhline(md, color='gray', linestyle='--', label=f'Mean Diff: {md:.2f}'); axes[1].axhline(md + 1.96 * sd, color='red', linestyle='--', label=f'+1.96 SD'); axes[1].axhline(md - 1.96 * sd, color='red', linestyle='--', label=f'-1.96 SD')
                        axes[1].set_xlabel("Average of GT and Prediction (%)"); axes[1].set_ylabel("Difference (GT - Prediction) (%)"); axes[1].set_title("Bland-Altman Plot"); axes[1].grid(True); axes[1].legend()
                        plt.suptitle(f"Performance Analysis for {run_name} (Test Set)"); plt.tight_layout(rect=[0, 0.03, 1, 0.95]); fig.savefig(plot_path); plt.close(fig); print(f"SUCCESS: Generated 'performance_analysis_plot.png'.")
                    else: print("WARNING: Empty prediction data. Skipping performance plot.")
            except Exception as e: print(f"ERROR: Could not generate performance plot for [{run_name}]. Reason: {e}")
        else:
            print("INFO: 'performance_analysis_plot.png' already exists. Skipping generation.")

        if not os.path.exists(metrics_path): # Only generate if missing
            print(f"INFO: 'evaluation_metrics.txt' missing. Generating...")
            try:
                with open(metrics_path, 'w') as f:
                    f.write(f"Evaluation Metrics for Run: {run_name}\n"); f.write("="*40 + "\n\n")
                    if run_name in segmentation_model_names: f.write("Evaluation Metrics (RMSE/MAE manually calculated from EF output only):\n")
                    else: f.write("Evaluation Metrics:\n")
                    f.write("Validation Set Metrics:\n"); f.write(f"  - RMSE: {val_metrics.get('rmse', np.nan):.4f}\n"); f.write(f"  - MAE: {val_metrics.get('mae', np.nan):.4f}\n"); f.write(f"  - Sklearn R^2: {val_r2_sklearn:.4f}\n\n")
                    f.write("Test Set Metrics:\n"); f.write(f"  - RMSE: {test_metrics.get('rmse', np.nan):.4f}\n"); f.write(f"  - MAE: {test_metrics.get('mae', np.nan):.4f}\n"); f.write(f"  - Sklearn R^2: {test_r2_sklearn:.4f}\n")
                print(f"SUCCESS: Generated 'evaluation_metrics.txt'.")
            except Exception as e: print(f"ERROR: Could not generate metrics file for [{run_name}]. Reason: {e}")
        else:
            print("INFO: 'evaluation_metrics.txt' already exists. Skipping generation.")

    except Exception as e: # Catch errors for this folder
        print(f"CRITICAL FAILURE processing folder '{run_name}'. Reason: {e}")
        error_count_this_run += 1
        folder_successfully_processed = False # Ensure failure

    finally: # Save state ONLY if all files now exist
        all_files_exist = (
            os.path.exists(metrics_path) and
            os.path.exists(plot_path) and
            os.path.exists(history_plot_path) and
            os.path.exists(y_true_path) and
            os.path.exists(y_pred_path) and
            os.path.exists(y_true_val_path) and
            os.path.exists(y_pred_val_path)
        )

        if folder_successfully_processed and all_files_exist:
            save_completed_folder(STATE_FILE_PATH, experiment_dir)
            newly_completed += 1
            print(f"--- Successfully processed and saved state for [{run_name}] ---")
        elif folder_successfully_processed and not all_files_exist:
             print(f"--- WARNING: Evaluation succeeded but some files still missing for [{run_name}]. Not marking as complete. ---")
             if error_count_this_run == 0: error_count_this_run += 1
        else:
             print(f"--- FAILED processing for [{run_name}]. Review errors above. Will retry on next run. ---")
        time.sleep(0.5)

# --- Final Summary ---
print("\n" + "="*50)
print("RESUMABLE DATA EXPORT & REGENERATION COMPLETE")
final_completed_folders = load_completed_folders(STATE_FILE_PATH)
total_completed = len(final_completed_folders)
print(f"Total Folders Found: {total_folders}")
print(f"Attempted processing in this run: {processed_in_this_run}")
print(f"Newly completed in this run: {newly_completed}")
print(f"Skipped (already complete): {skipped_already_done}")
print(f"Total completed (cumulative): {total_completed}")
print(f"Errors encountered in this run: {error_count_this_run}")
remaining_folders = total_folders - total_completed
if remaining_folders > 0: print(f"\nThere are still {remaining_folders} folders that failed."); print("Run this script again to retry them."); print("Keeping Colab runtime active.")
else:
    print("\nAll folders processed successfully!");
    if os.path.exists(STATE_FILE_PATH): print(f"You can delete the state file if desired: {STATE_FILE_PATH}")
    print("You can now use the 'Fast Title Updater' script anytime."); print("Unassigning Colab runtime..."); from google.colab import runtime; runtime.unassign()
print("="*50)

Mounted at /content/drive
--- Drive Mount Successful ---
Connecting to Kaggle datasets...
Using Colab cache for faster access to the 'heartdatabase' dataset.
Using Colab cache for faster access to the 'lvef-two-stream-tfrecord-files' dataset.
Using Colab cache for faster access to the 'lvef-i3d-regression-tfrecord-files' dataset.
Using Colab cache for faster access to the 'lvef-i3d-regression-tfrecord-v-s-files' dataset.
Dataset connection complete.

STARTING NON-DESTRUCTIVE RESUMABLE DATA EXPORT
This will ONLY generate files that are missing.
Progress tracked in: /content/drive/My Drive/lvef_data_export_progress.txt

Loaded 94 completed folders from state file.
Searching for model folders containing 'best_model.keras'...
Found 62 total model folder(s).

--- Checking Folder 1/62: [CNN_RNN Scratch-1] ---
INFO: Skipping folder - already marked as completed.

--- Checking Folder 2/62: [Fusion NC-1] ---
INFO: Skipping folder - already marked as completed.

--- Checking Folder 3/62: [Fusion